# 02 · Phase 2 — Profile and clean (raw → clean)

**Input:** `data/raw/` (read-only, verified in Phase 1) · **Output:** `data/clean/` (10 CSV tables + `manifest.json`)

**Principles.** Raw is never modified. Every cleaning rule is a *numbered assumption* (`G1…P5`, listed below), is counted when applied,
and is written to the clean manifest. Missing ≠ empty ≠ zero: `NULL` means "the source did not say", and nothing is imputed.

**Raw files needed (only these 4 kinds):**

| Source | File(s) | Used for |
|---|---|---|
| KEV | `kev/known_exploited_vulnerabilities.json` | the scope (1,391 CVEs) |
| NVD | `nvd/cve/CVE-*.json` (1,391 files) | description, CVSS, CWE |
| ATT&CK | `attack/enterprise-attack-16.1.json` | techniques, tactics |
| PKEV | `pkev/kev-07.28.2025_attack-16.1-enterprise.json` | CVE → technique mappings |

Schemas and licenses in `data/raw/` are not needed for cleaning.

In [1]:
import sys, json, platform
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "sources.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))
%load_ext autoreload
%autoreload 2
import pandas as pd
from IPython.display import display
from threatgraph import config, cleaning as C
from threatgraph.profiling import profile_frame
from threatgraph.manifest import read_manifest
from threatgraph.integrity import sha256_file
pd.set_option("display.max_columns", 30, "display.max_colwidth", 70, "display.width", 200, "display.max_rows", 100)
SRC = config.load_sources(); RAW = config.RAW_DIR; CLEAN = config.CLEAN_DIR
raw_manifest = read_manifest(config.MANIFEST_PATH)
RAW_MANIFEST_SHA_BEFORE = sha256_file(config.MANIFEST_PATH)

## 1 · Gate in: are the raw files still exactly what Phase 1 validated? (assumption G1)

In [2]:
n = C.verify_raw_inputs(RAW, raw_manifest)          # raises IntegrityError if any byte changed
assert json.loads(config.REPORT_PATH.read_text())["all_passed"], "Phase 1 gate not passed"
print(f"OK: {n} raw files match the Phase-1 manifest; Phase-1 gate = passed")

OK: 1401 raw files match the Phase-1 manifest; Phase-1 gate = passed


## 2 · The assumptions (read them: they are what you defend at the exam)

In [3]:
display(pd.Series(C.ASSUMPTIONS, name="assumption").to_frame())

,assumption
G1,Raw files are never modified. Cleaning starts only if every raw fi...
G2,Missing != empty. An empty/blank string becomes NULL (empty CSV ce...
G3,No cross-source imputation: a value from source A never fills a ga...
G4,Text: leading/trailing whitespace is stripped; nothing else is cha...
G5,Time: KEV dates are calendar dates (no zone). NVD timestamps have ...
G6,"Scope = the 1,391 CVEs in KEV. NVD/PKEV rows outside KEV would be ..."
K1,"knownRansomwareCampaignUse is kept as an enum {known, unknown}. 'u..."
K2,KEV 'cwes' = [] (CWE not provided) -> no rows in kev_cwe. Not fill...
K3,"Entries with dueDate < dateAdded (15, bulk load 2021-11-03) keep t..."
N1,One row per CVE. description_en = the English description (NULL if...


## 3 · Load raw (read-only) and profile it — the evidence behind the rules

In [4]:
raw = C.load_raw(RAW)
print({k: (len(v["vulnerabilities"]) if k == "kev" else len(v["mapping_objects"]) if k == "pkev" else len(v["objects"]) if k == "attack" else len(v)) for k, v in raw.items()})

{'kev': 1391, 'pkev': 1183, 'attack': 22905, 'nvd': 1391}


### 3.1 KEV — lists (`cwes`) and padded text

In [5]:
display(profile_frame(pd.DataFrame(raw['kev']['vulnerabilities'])))

,kind,rows,nulls,empty_lists,blank_strings,edge_whitespace,distinct
column,,,,,,,
cveID,text,1391,0,0,0,0,1391
vendorProject,text,1391,0,0,0,2,228
product,text,1391,0,0,0,5,564
vulnerabilityName,text,1391,0,0,0,20,1088
dateAdded,text,1391,0,0,0,0,319
shortDescription,text,1391,0,0,0,19,1307
requiredAction,text,1391,0,0,0,2,39
dueDate,text,1391,0,0,0,0,339
knownRansomwareCampaignUse,text,1391,0,0,0,0,2


### 3.2 PKEV — `comments` null, `references` is a list of messy strings

In [6]:
pk = pd.DataFrame(raw["pkev"]["mapping_objects"])
display(profile_frame(pk.drop(columns=["references"])))
refs = pd.Series([u for lst in pk.references for u in lst])
print("reference strings:", len(refs), "| with spaces inside:", int(refs.str.strip().str.contains(r"\s").sum()),
      "| not starting with http(s)://:", int((~refs.str.strip().str.match(r"https?://")).sum()))
display(refs[~refs.str.strip().str.match(r"https?://")].str.slice(0, 80).value_counts().head(6).to_frame("count"))

,kind,rows,nulls,empty_lists,blank_strings,edge_whitespace,distinct
column,,,,,,,
capability_id,text,1183,0,0,0,0,419
capability_description,text,1183,0,0,0,16,357
mapping_type,text,1183,0,0,0,0,3
attack_object_id,text,1183,0,0,0,0,155
attack_object_name,text,1183,0,0,0,0,153
capability_group,text,1183,0,0,0,0,36
comments,text,1183,5,0,0,268,453
status,text,1183,0,0,0,0,1
mapping_framework_version,text,1183,0,0,0,0,1


reference strings: 2243 | with spaces inside: 18 | not starting with http(s)://: 25


,count
Exploitation,6
of%20VMs%20compromising%20the%20host.,6
and%20obtain%20domain%20administrator%20access.,4
PHP%20code%20on%20susceptible%20targets.,3
well%20as%20through%20Cortex%20XDR.&text=Updated%20Sept.,2
that%20were%20listed%20in%20error.,2


### 3.3 NVD — which CVSS versions exist, which `type`, which `vulnStatus`

In [7]:
rows = []
for cve, r in raw["nvd"].items():
    m31 = r["metrics"].get("cvssMetricV31", [])
    rows.append({"cve": cve, "vulnStatus": r["vulnStatus"], "n_v31": len(m31), "has_primary_v31": any(m["type"] == "Primary" for m in m31),
                 "has_v2": bool(r["metrics"].get("cvssMetricV2")), "n_weakness_rows": sum(len(w["description"]) for w in r["weaknesses"]),
                 "n_en_desc": sum(d["lang"] == "en" for d in r["descriptions"])})
nvd_probe = pd.DataFrame(rows)
display(nvd_probe.drop(columns="cve").describe(include="all").T)
print("no v3.1 at all:", int((nvd_probe.n_v31 == 0).sum()), "| v3.1 but only a CNA ('Secondary') metric:", int(((nvd_probe.n_v31 > 0) & ~nvd_probe.has_primary_v31).sum()))
display(pd.crosstab(nvd_probe.vulnStatus, nvd_probe.has_primary_v31))

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
vulnStatus,1391,4,Analyzed,1069,NaN,NaN,NaN,NaN,NaN,NaN,NaN
n_v31,1391.0,NaN,NaN,NaN,1.797987,0.419176,0.0,2.0,2.0,2.0,3.0
has_primary_v31,1391,2,True,1254,NaN,NaN,NaN,NaN,NaN,NaN,NaN
has_v2,1391,2,True,937,NaN,NaN,NaN,NaN,NaN,NaN,NaN
n_weakness_rows,1391.0,NaN,NaN,NaN,1.813803,0.512277,1.0,2.0,2.0,2.0,5.0
n_en_desc,1391.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0


no v3.1 at all: 2 | v3.1 but only a CNA ('Secondary') metric: 135


has_primary_v31,False,True
vulnStatus,,
Analyzed,99,970
Deferred,38,278
Modified,0,2
Undergoing Analysis,0,4


### 3.4 ATT&CK — revoked / deprecated techniques

In [8]:
t = [o for o in raw["attack"]["objects"] if o["type"] == "attack-pattern"]
print(pd.Series(["revoked" if o.get("revoked") else "deprecated" if o.get("x_mitre_deprecated") else "active" for o in t]).value_counts().to_dict())

{'active': 656, 'revoked': 131, 'deprecated': 12}


## 4 · Clean (in memory first)

In [9]:
tables, log = C.clean_all(raw)
summary = pd.DataFrame({"rows": {k: len(v) for k, v in tables.items()}, "columns": {k: v.shape[1] for k, v in tables.items()}})
display(summary)

,rows,columns
kev_vulnerability,1391,12
kev_cwe,1328,2
nvd_cve,1391,13
nvd_cve_cwe,2523,5
attack_tactic,14,4
attack_technique,799,9
attack_technique_tactic,1051,2
pkev_mapping,1183,7
pkev_mapping_reference,2256,3


### 4.1 Which rules fired, and how often

In [10]:
rules = pd.DataFrame({"times": log.counts})
rules["assumption"] = [C.ASSUMPTIONS[k.split(".")[0]][:110] + "…" for k in rules.index]
display(rules)

,times,assumption
K1.ransomware_unknown,1100,"knownRansomwareCampaignUse is kept as an enum {known, unknown}. 'u..."
G4.stripped_values,559,Text: leading/trailing whitespace is stripped; nothing else is cha...
K1.ransomware_known,291,"knownRansomwareCampaignUse is kept as an enum {known, unknown}. 'u..."
K2.empty_cwes,148,KEV 'cwes' = [] (CWE not provided) -> no rows in kev_cwe. Not fill...
K3.due_before_added,15,"Entries with dueDate < dateAdded (15, bulk load 2021-11-03) keep t..."
N2.several_v31_metrics,1104,CVSS: only v3.1 goes into cvss_v31_*. Metric choice: the 'Primary'...
N4.deferred,316,vulnStatus is stored as given. 'Deferred' means NVD did not analys...
N3.noninformative,373,"Weaknesses: one row per distinct (CVE, CWE, source). 'NVD-CWE-noin..."
N2.no_v31,2,CVSS: only v3.1 goes into cvss_v31_*. Metric choice: the 'Primary'...
N2.v2_only,2,CVSS: only v3.1 goes into cvss_v31_*. Metric choice: the 'Primary'...


### 4.2 Before → after (four concrete cases)

In [11]:
k = next(v for v in raw["kev"]["vulnerabilities"] if not v["cwes"])
print("KEV raw   (K2: cwes = []):", {x: k[x] for x in ("cveID", "cwes", "knownRansomwareCampaignUse", "dueDate", "dateAdded")})
print("KEV clean:"); display(tables["kev_vulnerability"].query("cve_id == @k['cveID']")[["cve_id", "ransomware_use", "due_date", "date_added", "due_before_added"]])
print("          rows in kev_cwe for this CVE:", int((tables["kev_cwe"].cve_id == k["cveID"]).sum()), "<- zero rows, not a fake CWE")

sec = tables["nvd_cve"].query("cvss_v31_type == 'Secondary'").iloc[0]
print("\nNVD (N2): this CVE has only a CNA metric -> kept, type stored:", sec.cve_id, sec.cvss_v31_score, sec.cvss_v31_type, "| vulnStatus:", sec.vuln_status)
none31 = tables["nvd_cve"][tables["nvd_cve"].cvss_v31_score.isna()]
print("NVD (N2): no v3.1 -> NULL, v2 kept separately (never converted):"); display(none31[["cve_id", "cvss_v31_score", "cvss_v2_score", "vuln_status"]])

glued = next(u for lst in pk.references for u in lst if len(u.split()) > 1 and all(x.rstrip(";").startswith("http") for x in u.split()))
print("\nPKEV (P3) raw string:", repr(glued[:140]))
print("PKEV (P3) clean rows:"); display(tables["pkev_mapping_reference"][tables["pkev_mapping_reference"].reference.isin([x.rstrip(";") for x in glued.split()])].drop_duplicates("reference"))
print("PKEV (P3) kept verbatim and flagged:"); display(tables["pkev_mapping_reference"].query("not is_http_url").drop_duplicates("reference").head(5))

print("\nATT&CK (T2): a revoked technique stays, with a status:"); display(tables["attack_technique"].query("status == 'revoked'").head(2).drop(columns=["description"]))

KEV raw   (K2: cwes = []): {'cveID': 'CVE-2025-43200', 'cwes': [], 'knownRansomwareCampaignUse': 'Unknown', 'dueDate': '2025-07-07', 'dateAdded': '2025-06-16'}
KEV clean:


,cve_id,ransomware_use,due_date,date_added,due_before_added
1374,CVE-2025-43200,unknown,2025-07-07,2025-06-16,False


          rows in kev_cwe for this CVE: 0 <- zero rows, not a fake CWE

NVD (N2): this CVE has only a CNA metric -> kept, type stored: CVE-2010-2568 7.8 Secondary | vulnStatus: Deferred
NVD (N2): no v3.1 -> NULL, v2 kept separately (never converted):


,cve_id,cvss_v31_score,cvss_v2_score,vuln_status
17,CVE-2009-2055,NaN,4.3,Deferred
144,CVE-2015-0666,NaN,7.8,Deferred



PKEV (P3) raw string: ' https://thehackernews.com/2025/07/google-patches-critical-zero-day-flaw.html https://www.menlosecurity.com/blog/chrome-zero-day-why-browser'
PKEV (P3) clean rows:


,mapping_id,reference,is_http_url
138,94,https://thehackernews.com/2025/07/google-patches-critical-zero-day...,True
139,94,https://www.menlosecurity.com/blog/chrome-zero-day-why-browser-sec...,True


PKEV (P3) kept verbatim and flagged:


,mapping_id,reference,is_http_url
477,237,and%20obtain%20domain%20administrator%20access.,False
612,319,[],False
1367,682,chrome-extension://efaidnbmnnnibpcajpcglclefindmkaj/https://www.ci...,False
1543,773,PHP%20code%20on%20susceptible%20targets.,False
1607,800,Exploitation,False



ATT&CK (T2): a revoked technique stays, with a status:


,technique_id,stix_id,name,is_subtechnique,parent_technique_id,status,created,modified
4,T1002,attack-pattern--b9f5dbe2-4c55-4fc5-af2e-d42c1d182ec4,Data Compressed,False,<NA>,revoked,2017-05-31 21:30:19.338000+00:00,2022-04-25 14:00:00.188000+00:00
14,T1004,attack-pattern--514ede4c-78b3-4d78-a38b-daddf6217a79,Winlogon Helper DLL,False,<NA>,revoked,2017-05-31 21:30:20.148000+00:00,2022-04-25 14:00:00.188000+00:00


## 5 · Write `data/clean/` + manifest

In [12]:
meta = {"raw_manifest_sha256": RAW_MANIFEST_SHA_BEFORE, "sources_digest_sha256": config.sources_digest(),
        "cleaning_code_sha256": sha256_file(ROOT / "src" / "threatgraph" / "cleaning.py"),
        "environment": {"python": platform.python_version(), "pandas": pd.__version__}}
manifest = C.write_clean(tables, CLEAN, log, meta)
display(pd.DataFrame(manifest["tables"]).T[["file", "rows", "bytes", "sha256"]].assign(sha256=lambda d: d.sha256.str.slice(0, 16) + "…"))

,file,rows,bytes,sha256
kev_vulnerability,kev_vulnerability.csv,1391,706216,fc0c9410db476d57…
kev_cwe,kev_cwe.csv,1328,29558,b84bc6036a202b16…
nvd_cve,nvd_cve.csv,1391,724852,435b99c03249cd74…
nvd_cve_cwe,nvd_cve_cwe.csv,2523,148888,51ec143965c24292…
attack_tactic,attack_tactic.csv,14,1285,7e2e881702e4765d…
attack_technique,attack_technique.csv,799,1195845,912ea691422248ae…
attack_technique_tactic,attack_technique_tactic.csv,1051,16182,3008af112e5ab1b4…
pkev_mapping,pkev_mapping.csv,1183,518104,6b76edf9253c3e84…
pkev_mapping_reference,pkev_mapping_reference.csv,2256,224856,c9111e106ef37a8d…


## 6 · Gate out: raw is still untouched

In [13]:
assert sha256_file(config.MANIFEST_PATH) == RAW_MANIFEST_SHA_BEFORE
print("raw files verified again:", C.verify_raw_inputs(RAW, read_manifest(config.MANIFEST_PATH)), "match the Phase-1 manifest -> cleaning did not touch raw")

raw files verified again: 1401 match the Phase-1 manifest -> cleaning did not touch raw


**Next:** `03_validate_clean_data.ipynb` checks the clean tables (schema, keys, domains, semantics, independent recount, provenance)
*before* any database is built. Phase 3 (Django) does not start until that gate passes.